# SGA-2025 + ssl-legacysurvey Tutorial

This notebook walks through the end-to-end workflow for running the
[ssl-legacysurvey](https://github.com/georgestein/ssl-legacysurvey) self-supervised
contrastive-learning pipeline on a small subset of SGA-2025 galaxy mosaics.

**Steps covered:**
1. Select N galaxies from the SGA-2025 sample
2. Build the reference catalog (size / isolation cuts)
3. Build HDF5 input files from the on-disk FITS cutouts
4. Run MoCo v2 inference to get embeddings
5. Inspect the results

**Requirements:**
- SGAML Jupyter kernel (see `etc/README.md`)
- Environment variables `SGA_DIR` and `SGA_DATA_DIR` set (e.g., via `source bin/SGA2025/SGA2025-env`)
- Pre-trained checkpoint `resnet50.ckpt` (download from the ssl-legacysurvey repo)

## 1. Setup

In [ ]:
import os
import numpy as np
from astropy.table import vstack

from SGA.SGA import read_sample
from SGA.ssl import build_ssl_legacysurvey_refcat, build_ssl_legacysurvey, ssl_match
from SGA.logger import log

# Verify required environment variables are set
for var in ('SGA_DIR', 'SGA_DATA_DIR'):
    val = os.getenv(var)
    if val is None:
        raise EnvironmentError(f'{var} is not set. Run: source bin/SGA2025/SGA2025-env')
    print(f'{var} = {val}')

In [ ]:
# --- User-settable parameters ---
N_GALAXIES   = 100          # number of galaxies to select
SEED         = 42           # random seed for reproducibility
SSL_VERSION  = 'v3'         # 'v1'/'v2' = grz only; 'v3'/'v4' = grz + griz
CHECKPOINT   = 'resnet50.ckpt'   # path to pre-trained MoCo v2 checkpoint
OUTDIR       = '/tmp/sga-ssl-tutorial'  # output directory for HDF5 files and results
CUTOUTDIR    = os.environ['SGA_DATA_DIR']  # root of on-disk FITS cutouts

os.makedirs(OUTDIR, exist_ok=True)
print(f'Output directory: {OUTDIR}')

## 2. Load the SGA-2025 sample

`read_sample` returns two tables: `sample` (one row per galaxy group, GROUP_PRIMARY only)
and `fullsample` (all group members including satellites).

In [ ]:
# ssl v3/v4 uses the parent catalog (final_sample=False) spanning dr9-north + dr11-south
sample_n, fullsample_n = read_sample(final_sample=False, region='dr9-north')
sample_s, fullsample_s = read_sample(final_sample=False, region='dr11-south')

sample     = vstack([sample_n,     sample_s])
fullsample = vstack([fullsample_n, fullsample_s])
del sample_n, sample_s, fullsample_n, fullsample_s

print(f'Primary sample:  {len(sample):,} groups')
print(f'Full sample:     {len(fullsample):,} objects (including satellites)')

## 3. Build the ssl reference catalog

Apply size, isolation, and imaging-quality cuts to define the pool of
objects that can serve as *reference* galaxies for the ssl pipeline.
`build_ssl_legacysurvey_refcat` returns:
- `refcat` — objects that pass all reference-galaxy cuts
- `cat`    — objects that pass the looser candidate cuts

In [ ]:
refcat, cat = build_ssl_legacysurvey_refcat(sample, fullsample, ssl_version=SSL_VERSION)
print(f'Reference catalog: {len(refcat):,} objects')
print(f'Candidate catalog: {len(cat):,} objects')

## 4. Select N galaxies for the tutorial

For this tutorial we draw a small random subset so the HDF5 build and
inference steps run quickly.  Modify `select_sample` to apply whatever
science-driven selection you need (e.g., magnitude, colour, environment).

In [ ]:
def select_sample(cat, n=100, seed=42):
    """Return a random draw of n objects from cat.

    Parameters
    ----------
    cat : astropy.table.Table
        Input catalog (e.g. the refcat or candidate cat from build_ssl_legacysurvey_refcat).
    n : int
        Number of objects to draw.  If n >= len(cat), returns the full catalog.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    astropy.table.Table
        Sub-table of n randomly selected rows.
    """
    rng = np.random.default_rng(seed)
    n   = min(n, len(cat))
    idx = rng.choice(len(cat), size=n, replace=False)
    idx.sort()  # preserve spatial ordering
    return cat[idx]


sub = select_sample(cat, n=N_GALAXIES, seed=SEED)
print(f'Selected {len(sub)} galaxies for the tutorial')
sub[['SGAID', 'GROUP_NAME', 'GROUP_RA', 'GROUP_DEC', 'DIAM', 'BANDS']][:5]

## 5. Build HDF5 input files

`build_ssl_legacysurvey` locates the rescaled FITS cutout for each object,
selects the requested band planes, and packs everything into chunked HDF5
files ready for inference.

In [ ]:
build_ssl_legacysurvey(
    sub,
    refcat,
    ssl_version=SSL_VERSION,
    cutoutdir=CUTOUTDIR,
    outdir=OUTDIR,
    cutout_regions=['dr9-north', 'dr11-south'],
    verbose=True,
    overwrite=True,
)

In [ ]:
import glob
hdf5_files = sorted(glob.glob(os.path.join(OUTDIR, '*.hdf5')))
print(f'HDF5 files written: {len(hdf5_files)}')
for f in hdf5_files:
    print(' ', f)

## 6. Inspect HDF5 contents

In [ ]:
import h5py
import matplotlib.pyplot as plt

if hdf5_files:
    with h5py.File(hdf5_files[0], 'r') as f:
        print('Keys:', list(f.keys()))
        images = f['images'][:]   # (N, nband, H, W)
        ras    = f['ra'][:]
        decs   = f['dec'][:]
        rows   = f['row'][:]      # SGAID
        print(f'Image array shape: {images.shape}')
        print(f'RA range: {ras.min():.3f} – {ras.max():.3f}')

    # Show the first 9 galaxies (median-scale each for display)
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for ax, img, ra, dec in zip(axes.flat, images[:9], ras[:9], decs[:9]):
        rgb = np.dstack([img[2], img[1], img[0]])  # z, r, g → RGB
        lo, hi = np.percentile(rgb, [1, 99])
        rgb = np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)
        ax.imshow(rgb, origin='lower')
        ax.set_title(f'({ra:.3f}, {dec:.3f})', fontsize=7)
        ax.axis('off')
    plt.suptitle('First 9 tutorial galaxies (grz)', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('No HDF5 files found — check CUTOUTDIR and that cutouts exist on disk.')

## 7. Run ssl_match inference

`ssl_match` loads the pre-trained MoCo v2 backbone, encodes every galaxy
in the HDF5 file, and writes two outputs to `output_dir`:
- `*_output.txt` — per-galaxy embedding vectors
- `*_umap.npy`   — 2-d UMAP projection for visualisation

In [ ]:
results_dir = os.path.join(OUTDIR, 'results')
os.makedirs(results_dir, exist_ok=True)

for path in hdf5_files:
    print(f'Running ssl_match on {os.path.basename(path)} ...')
    ssl_match(
        path,
        checkpoint_path=CHECKPOINT,
        output_dir=results_dir,
        similarity=True,
        threshold=0.5,
    )

## 8. Inspect results

The UMAP projection gives a 2-d view of the embedding space where
morphologically similar galaxies cluster together.

In [ ]:
umap_files = sorted(glob.glob(os.path.join(results_dir, '*_umap.npy')))
print(f'UMAP files: {umap_files}')

if umap_files:
    umap = np.load(umap_files[0])
    print(f'UMAP shape: {umap.shape}')   # (N, 2)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(umap[:, 0], umap[:, 1], s=8, alpha=0.7, linewidths=0)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title(f'ssl-legacysurvey embeddings — {len(umap)} galaxies')
    plt.tight_layout()
    plt.show()
else:
    print('No UMAP output found — did ssl_match run successfully?')